# Cbow Embeding Berita Preprocesed

In [1]:
%%capture
!pip install plotly
!pip install --upgrade gensim

## Ringkasan Alur
- Import pustaka yang dibutuhkan
- Baca dataset `hasil_preprocessing_berita.csv`
- Pembersihan teks (lowercase, hapus tanda baca/tag/digit)
- Tokenisasi dan pembuatan `corpus`
- Latih Word2Vec (CBOW) untuk embedding kata
- Hitung mean embedding per dokumen (rata-rata vektor kata)
- Bentuk DataFrame fitur `f1..f56` dan tambahkan label `spam`
- (Opsional) Simpan hasil ke CSV


In [2]:
from gensim.models import Word2Vec, FastText
import pandas as pd
import re

from sklearn.decomposition import PCA

from matplotlib import pyplot as plt
import plotly.graph_objects as go

import numpy as np

import warnings
warnings.filterwarnings('ignore')

# Load dataset TF-IDF berita dari file CSV
# Pastikan file 'hasil_tfidf_berita.csv' ada di direktori kerja notebook ini

df = pd.read_csv('hasil_preprocessing_berita.csv')

In [3]:
from gensim.models import Word2Vec

In [4]:
import numpy as np

class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [5]:
# Bangun korpus token dari teks hasil preprocessing: setiap baris -> list kata
# Gunakan kolom 'hasil_preprocessing' jika ada, jika tidak cari alternatif umum
import ast

text_col = 'hasil_preprocessing' if 'hasil_preprocessing' in df.columns else None
if text_col is None:
    possible_text_cols = ['clean', 'text', 'preprocessed', 'kalimat', 'sentence']
    text_col = next((c for c in possible_text_cols if c in df.columns), None)
if text_col is None:
    raise RuntimeError("Kolom teks tidak ditemukan. Pastikan 'hasil_preprocessing' atau kolom teks lain tersedia di CSV.")

# Parser aman: dukung format list-string seperti "['kata', 'kata2']" atau string biasa
def to_tokens(value):
    if isinstance(value, list):
        return [str(tok).lower() for tok in value if str(tok).strip()]
    if isinstance(value, str):
        s = value.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(tok).lower() for tok in parsed if str(tok).strip()]
            except Exception:
                pass
        return [tok for tok in s.lower().split() if tok]
    return []

texts = df[text_col].fillna("").tolist()
corpus = [tokens for tokens in (to_tokens(v) for v in texts) if tokens]

if len(corpus) == 0:
    raise RuntimeError("Corpus kosong: kolom teks ada tetapi tidak berisi token valid.")

# Contoh token dokumen pertama
corpus[0:1]

[['kompascom',
  'alam',
  'senang',
  'kabar',
  'alami',
  'keluarga',
  'timnas',
  'indonesia',
  'hadir',
  'langsung',
  'king',
  'abdullah',
  'sports',
  'city',
  'jeddah',
  'saksi',
  'juang',
  'skuad',
  'garuda',
  'lawan',
  'timnas',
  'arab',
  'saudi',
  'timnas',
  'indonesia',
  'hadap',
  'timnas',
  'arab',
  'saudi',
  'laga',
  'perdana',
  'grup',
  'b',
  'ronde',
  'empat',
  'kualifikasi',
  'piala',
  'dunia',
  'zona',
  'asia',
  'duel',
  'timnas',
  'indonesia',
  'vs',
  'arab',
  'saudi',
  'langsung',
  'king',
  'abdullah',
  'sports',
  'city',
  'jeddah',
  'rabu',
  'kamis',
  'wib',
  'lapang',
  'suporter',
  'indonesia',
  'berbondongbondong',
  'hadir',
  'stadion',
  'dukung',
  'langsung',
  'juang',
  'timnas',
  'indonesia',
  'kecuali',
  'keluarga',
  'main',
  'sayang',
  'lapor',
  'laku',
  'senang',
  'aman',
  'keluarga',
  'main',
  'kabar',
  'istri',
  'thom',
  'haye',
  'bibeche',
  'riva',
  'hamil',
  'enam',
  'alami',
  '

## Tokenisasi dan Pelatihan Word2Vec (CBOW)
- Tokenisasi setiap baris `clean` menjadi list kata → `corpus`
- Latih model Word2Vec dengan default CBOW (`sg=0`) dan `vector_size=56`
- Hasil: embedding vektor untuk setiap kata di vocabulary


In [6]:
df.shape

(1432, 3)

In [7]:
# Latih model Word2Vec (CBOW by default: sg=0) dengan ukuran vektor 56
# Menggunakan korpus token yang dibangun dari fitur TF-IDF (nilai > 0)
model = Word2Vec(corpus, min_count=1, vector_size=56)

In [8]:
# (Opsional) contoh eksplorasi embeddings kata jika diperlukan
# model.wv.most_similar('eric')
# model.wv.most_similar_cosmul(positive=['phone', 'number'], negative=['call'])
# model.wv.doesnt_match("phone number prison cell".split())

# Simpan embeddings kata yang dilatih
filename = 'berita_embd.txt'
model.wv.save_word2vec_format(filename, binary=False)

In [9]:
# Mean embedding per dokumen: rata-rata vektor kata dari token TF-IDF
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
# Gabungkan token menjadi string kalimat agar tokenizer bekerja sama seperti sebelumnya
joined_docs = [" ".join(tokens) for tokens in corpus]
mean_embedded = mean_embedding_vectorizer.fit_transform(joined_docs)

In [10]:
# Simpan vektor dokumen ke kolom 'array'
df['array']=list(mean_embedded)

## Rata-rata Embedding per Dokumen
- Gunakan `MeanEmbeddingVectorizer` untuk merata-ratakan vektor kata per dokumen
- Jika dokumen tidak punya kata di vocab, isi vektor nol berdimensi 56


In [11]:
df.head(5)

,isi,hasil_preprocessing,kategori,array
0,KOMPAS.com -Pengalaman tak menyenangkan kabarn...,"['kompascom', 'alam', 'senang', 'kabar', 'alam...",Bola,"[-0.05376743, 0.186151, 1.1708711, -0.11618368..."
1,"JAKARTA, KOMPAS.com –Tiktokers Figha Lesmana m...","['jakarta', 'kompascom', 'tiktokers', 'figha',...",Megapolitan,"[-0.13584949, -0.043743454, 0.5880184, 0.16620..."
2,KOMPAS.com -Wakil Presiden Direktur PT Toyota ...,"['kompascom', 'wakil', 'presiden', 'direktur',...",Money,"[-0.1138674, -0.12919497, 0.5343833, 0.2167285..."
3,"JAKARTA, KOMPAS.com- Menteri Koordinator Bidan...","['jakarta', 'kompascom', 'menteri', 'koordinat...",Nasional,"[-0.13909662, -0.050422728, 0.55936563, 0.2399..."
4,"JAKARTA, KOMPAS.com- Menteri Pertanian (Mentan...","['jakarta', 'kompascom', 'menteri', 'tani', 't...",Nasional,"[-0.12832472, -0.20336348, 0.53823465, 0.23120..."


In [12]:
df['embedding_length'] = df['array'].str.len()

In [13]:
print(df['embedding_length'])

0       56
1       56
2       56
3       56
4       56
        ..
1427    56
1428    56
1429    56
1430    56
1431    56
Name: embedding_length, Length: 1432, dtype: int64


## Bentuk DataFrame Fitur f1..f56 dan Tambah Label
- Ekstrak setiap dimensi embedding ke kolom `f1..f56`
- Tambahkan label `spam` dari dataset asli


In [14]:
df.shape

(1432, 5)

In [15]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

print(embedding_df)

            f1        f2        f3        f4        f5        f6        f7  \
0    -0.053767  0.186151  1.170871 -0.116184  0.361449 -1.366470 -0.261355   
1    -0.135849 -0.043743  0.588018  0.166201  0.207742 -0.827346  0.174960   
2    -0.113867 -0.129195  0.534383  0.216728  0.210930 -0.799077  0.063506   
3    -0.139097 -0.050423  0.559366  0.239994  0.240890 -0.669791  0.006320   
4    -0.128325 -0.203363  0.538235  0.231204  0.152997 -0.758839  0.122503   
...        ...       ...       ...       ...       ...       ...       ...   
1427 -0.082741 -0.006145  0.511954  0.107219  0.176803 -0.709393  0.041627   
1428 -0.120481 -0.329801  0.518664  0.201834  0.056005 -0.671259  0.249384   
1429 -0.071568 -0.007723  0.750166  0.005248  0.183865 -0.993290 -0.026059   
1430 -0.130158 -0.198836  0.629804  0.239695  0.203332 -1.002255  0.187142   
1431 -0.168589 -0.180674  0.567708  0.236988  0.241342 -0.864465  0.128030   

            f8        f9       f10  ...       f47       f48    

In [16]:
# Gunakan label kategori dari dataset TF-IDF berita
embedding_df['kategori'] = df['kategori'].values  

## Simpan Hasil ke CSV (Opsional)
Simpan `embedding_df` ke file CSV untuk digunakan di proses selanjutnya.


In [17]:
# Simpan DataFrame fitur dokumen ke CSV (opsional)
embedding_df.to_csv('berita_doc_embeddings.csv', index=False, encoding='utf-8')
print('Disimpan ke berita_doc_embeddings.csv')


Disimpan ke berita_doc_embeddings.csv


In [18]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f48,f49,f50,f51,f52,f53,f54,f55,f56,kategori
0,-0.053767,0.186151,1.170871,-0.116184,0.361449,-1.366470,-0.261355,-0.653515,0.025357,-1.032438,...,-0.222772,-0.502720,-0.086070,0.434817,1.522519,-0.405560,-0.982857,1.112036,-0.496941,Bola
1,-0.135849,-0.043743,0.588018,0.166201,0.207742,-0.827346,0.174960,-0.735400,-0.147986,-0.454669,...,0.052289,-0.179641,-0.004081,0.347398,0.691230,-0.437728,-0.247807,0.480904,-0.539126,Megapolitan
2,-0.113867,-0.129195,0.534383,0.216728,0.210930,-0.799077,0.063506,-0.707487,-0.274802,-0.537461,...,0.268831,-0.323963,-0.047821,0.297053,0.735117,-0.120526,-0.260022,0.443468,-0.338207,Money
3,-0.139097,-0.050423,0.559366,0.239994,0.240890,-0.669791,0.006320,-0.845639,-0.285780,-0.589436,...,0.142264,-0.164130,0.139959,0.268682,0.795615,-0.229748,-0.284800,0.572246,-0.501308,Nasional
4,-0.128325,-0.203363,0.538235,0.231204,0.152997,-0.758839,0.122503,-0.735269,-0.300734,-0.478442,...,0.256099,-0.426563,-0.047382,0.313714,0.828917,-0.036590,-0.252309,0.409118,-0.259371,Nasional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427,-0.082741,-0.006145,0.511954,0.107219,0.176803,-0.709393,0.041627,-0.546463,-0.118133,-0.431986,...,0.057599,-0.191931,-0.001224,0.246861,0.629920,-0.309519,-0.285527,0.440452,-0.388027,Entertainment
1428,-0.120481,-0.329801,0.518664,0.201834,0.056005,-0.671259,0.249384,-0.773642,-0.278334,-0.358817,...,0.201657,-0.480499,0.003577,0.336896,0.949529,0.089837,-0.284194,0.347200,-0.169617,Otomotif
1429,-0.071568,-0.007723,0.750166,0.005248,0.183865,-0.993290,-0.026059,-0.610252,-0.087622,-0.624097,...,0.030981,-0.382054,-0.125013,0.370870,1.025518,-0.199058,-0.553398,0.600611,-0.319553,Tekno
1430,-0.130158,-0.198836,0.629804,0.239695,0.203332,-1.002255,0.187142,-0.823035,-0.269155,-0.557059,...,0.236720,-0.351000,-0.018492,0.383732,0.774234,-0.337538,-0.255285,0.515943,-0.525235,Otomotif


In [19]:
embedding_df.shape

(1432, 57)